# Causal Light-health detection and realistic routing

Notebook 05 showed that a Light-independent fallback can help when an oracle tells us exactly when Light is faulty. This notebook removes that perfect-information assumption. A causal detector sees only the current and previous Light readings and decides whether to keep the Temperature + Light + CO₂ primary model or route to the Temperature + CO₂ fallback.

The occupancy models are not retrained here. The new component is the detector and its routing decision.

## 1. Experimental design

Detector settings are selected with simulated episodes inside five chronological training-validation folds. Only after the settings are fixed are new episodes inserted into copies of Test 1 and Test 2. Original source files remain unchanged.

| Simulated condition | What happens to Light? | Expected observability |
|---|---|---|
| Missing | Readings become unavailable for 15 or 60 minutes | Directly observable |
| Frozen at current value | The last value is repeated | Observable only after enough unchanged readings |
| Fixed low | Light is fixed at the training 5th percentile | Difficult: normal darkness can look identical |
| Fixed high | Light is fixed at the training 95th percentile | Potentially observable as an unusually constant high value |
| Out of range high | Light exceeds the training maximum by one training standard deviation | Directly observable by a range rule |
| Positive linear bias | An additive error grows from zero to +1 training standard deviation | Difficult if values stay plausible |
| Negative linear bias | An additive error grows from zero to −1 training standard deviation | Sometimes observable after crossing a low bound |

Occupied-dark and unoccupied-lit conditions are excluded. They may represent correct Light measurements under changed room behavior, not technical sensor failures.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULT_DIR = PROJECT_ROOT / "models" / "fault_detection"
required = {
    "cv": RESULT_DIR / "detector_cv_results.csv",
    "detection": RESULT_DIR / "heldout_detection_metrics.csv",
    "routing": RESULT_DIR / "heldout_routing_metrics.csv",
    "trace": RESULT_DIR / "example_episode_trace.csv",
    "metadata": RESULT_DIR / "metadata.json",
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run `python -m sensorbudget.robustness.fault_detection` first. "
        + f"Missing: {missing}"
    )

cv = pd.read_csv(required["cv"])
detection = pd.read_csv(required["detection"])
routing = pd.read_csv(required["routing"])
trace = pd.read_csv(required["trace"], parse_dates=["date"])
metadata = json.loads(required["metadata"].read_text(encoding="utf-8"))

PLOTLY_TEMPLATE = "plotly_white"
SPLIT_LABELS = {"test_1": "Test 1", "test_2": "Test 2"}
SCENARIO_LABELS = {
    "clean": "Clean data (no injected fault)",
    "missing": "Missing Light",
    "stuck_current": "Frozen at current value",
    "stuck_low": "Fixed at training 5th percentile (low)",
    "stuck_high": "Fixed at training 95th percentile (high)",
    "out_of_range_high": "Above training range",
    "linear_bias_positive": "Positive linear calibration bias",
    "linear_bias_negative": "Negative linear calibration bias",
}
STRATEGY_LABELS = {
    "primary_only": "Primary only",
    "fallback_only": "Fallback only",
    "oracle_routing": "Oracle routing",
    "detector_routing": "Detector routing",
}
for frame in (detection, routing, trace):
    frame["scenario_label"] = frame["scenario"].map(SCENARIO_LABELS)
detection["split_label"] = detection["split"].map(SPLIT_LABELS)
routing["split_label"] = routing["split"].map(SPLIT_LABELS)
routing["strategy_label"] = routing["strategy"].map(STRATEGY_LABELS)
print(f"Loaded {len(detection)} detector evaluations and {len(routing)} routing evaluations.")

Loaded 58 detector evaluations and 232 routing evaluations.


## 2. Training-selected detector

The detector combines missingness, trailing-window constancy, training-range, and abrupt-change rules. Candidates first had to keep the mean chronological-validation false-positive rate at or below 5%; selection then maximized mean detection F1.

In [2]:
selected = cv.loc[cv["selected"]].iloc[0]
selected_table = pd.DataFrame(
    {
        "Parameter": [
            "Trailing stuck window",
            "Allowed variation",
            "Lower range quantile",
            "Upper range quantile",
            "Abrupt-change quantile",
        ],
        "Selected value": [
            f"{int(selected['stuck_window'])} minutes",
            f"{selected['stuck_tolerance_lux']:.1f} lx",
            f"{selected['range_low_quantile']:.3f}",
            f"{selected['range_high_quantile']:.3f}",
            f"{selected['abrupt_change_quantile']:.3f}",
        ],
    }
)
display(selected_table)

metric_labels = {
    "detection_precision_mean": "Precision",
    "detection_recall_mean": "Recall",
    "detection_f1_mean": "F1",
    "false_positive_rate_mean": "False-positive rate",
}
values = pd.Series({label: selected[column] for column, label in metric_labels.items()})
errors = pd.Series(
    {label: selected[column.replace("_mean", "_std")] for column, label in metric_labels.items()}
)
metric_colors = {
    "Precision": "#457b9d",
    "Recall": "#457b9d",
    "F1": "#2a9d8f",
    "False-positive rate": "#e76f51",
}
fig = go.Figure()
for label in values.index:
    color = metric_colors[label]
    fig.add_trace(
        go.Bar(
            x=[values[label]],
            y=[label],
            orientation="h",
            customdata=[[errors[label]]],
            error_x={
                "type": "data",
                "array": [errors[label]],
                "visible": True,
                "color": color,
                "thickness": 1.5,
            },
            marker_color=color,
            showlegend=False,
            hovertemplate=(
                "%{y}<br>Mean: %{x:.3f}"
                "<br>Standard deviation: %{customdata[0]:.3f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Selected detector performance in chronological training validation",
    xaxis_title="Mean metric value (whisker = ±1 standard deviation)",
    xaxis={"range": [0, 1.01]},
    yaxis_title="Metric",
    height=400,
)
fig.show()

,Parameter,Selected value
0,Trailing stuck window,20 minutes
1,Allowed variation,0.0 lx
2,Lower range quantile,0.001
3,Upper range quantile,0.999
4,Abrupt-change quantile,1.000


**Conclusion.** The selected detector uses a strict 20-minute, zero-variation stuck rule and central 99.8% training range. Its mean validation F1 is only 0.398, which already warns that plausible stuck values and calibration bias are substantially harder than explicit missingness. The error bars show one standard deviation across folds, fault types, episode lengths, and placements; their size reflects systematic differences between easy and difficult failure modes rather than measurement noise alone. The 1.15% validation false-positive rate satisfies the configured safety ceiling.

## 3. What a detector sees during an episode

The shaded interval is the injected ground-truth fault. Red crosses mark rows where any detector rule fires. Because the detector is causal, a stuck episode cannot be recognized until enough unchanged readings have accumulated.

In [3]:
example_scenarios = ["missing", "stuck_high"]
fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=tuple(SCENARIO_LABELS[value] for value in example_scenarios),
    shared_xaxes=False,
)
for row_number, scenario in enumerate(example_scenarios, start=1):
    rows = trace.loc[(trace["split"] == "test_1") & (trace["scenario"] == scenario)].copy()
    fig.add_trace(
        go.Scatter(
            x=rows["date"],
            y=rows["Light"],
            mode="lines+markers",
            name="Reported Light",
            line={"color": "#457b9d"},
            showlegend=row_number == 1,
            hovertemplate="%{x}<br>Light: %{y:.1f} lx<extra></extra>",
        ),
        row=row_number,
        col=1,
    )
    alerts = rows.loc[rows["detected_light_fault"]].copy()
    marker_height = rows["Light"].dropna().max()
    alerts["marker_height"] = marker_height
    fig.add_trace(
        go.Scatter(
            x=alerts["date"],
            y=alerts["marker_height"],
            mode="markers",
            name="Detector alert",
            marker={"color": "#d62828", "symbol": "x", "size": 9},
            showlegend=row_number == 1,
            hovertemplate="%{x}<br>Detector alert<extra></extra>",
        ),
        row=row_number,
        col=1,
    )
    fault_rows = rows.loc[rows["true_light_fault"]]
    fig.add_vrect(
        x0=fault_rows["date"].min(),
        x1=fault_rows["date"].max(),
        fillcolor="#f4a261",
        opacity=0.18,
        line_width=0,
        row=row_number,
        col=1,
    )
fig.update_yaxes(title_text="Light (lx)")
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Example 15-minute Test 1 fault episodes and causal alerts",
    height=650,
)
fig.show()

**Conclusion.** Missing readings are flagged on the first unavailable row. A fixed high value requires accumulated evidence from the trailing window, so detection is delayed. Red alerts outside the shaded interval are false positives caused by naturally unusual or constant Light behavior.

## 4. Held-out detection precision and recall

Recall asks what share of injected fault rows are detected. Precision asks what share of all alerts actually fall inside the injected episode. Short episodes make precision especially sensitive to alerts elsewhere in the much longer held-out period.

In [4]:
summary = (
    detection.loc[detection["scenario"] != "clean"]
    .groupby(["split", "scenario_label"], as_index=False)[
        ["detection_precision", "detection_recall"]
    ].mean()
)
scenario_order = [
    label for key, label in SCENARIO_LABELS.items() if key != "clean"
]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"), shared_yaxes=True)
metric_colors = {"detection_precision": "#457b9d", "detection_recall": "#2a9d8f"}
metric_names = {"detection_precision": "Precision", "detection_recall": "Recall"}
for column, split in enumerate(["test_1", "test_2"], start=1):
    split_rows = summary.loc[summary["split"] == split]
    # Draw Precision first so Recall appears above it in each group.
    for metric in ["detection_precision", "detection_recall"]:
        rows = split_rows.copy()
        labels = rows[metric].map(lambda value: f"{value:.3f}")
        fig.add_trace(
            go.Bar(
                x=rows[metric],
                y=rows["scenario_label"],
                orientation="h",
                name=metric_names[metric],
                marker_color=metric_colors[metric],
                text=labels,
                textposition="outside",
                cliponaxis=False,
                showlegend=column == 1,
                hovertemplate=(
                    metric_names[metric]
                    + "<br>%{y}<br>Value: %{x:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fig.update_xaxes(range=[0, 1.08], title_text="Detection metric")
fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(scenario_order)))
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Detection quality by simulated Light condition",
    barmode="group",
    legend={"traceorder": "reversed"},
    height=650,
    margin={"l": 285, "r": 40},
)
fig.show()

**Conclusion.** Missing and extreme-high readings achieve perfect recall, but precision remains low because held-out Light behavior triggers alerts outside the short injected episodes. Fixed-low darkness and positive calibration bias are not detected. This is an honest observability limit: normal darkness and plausible biased values do not provide enough evidence by themselves.

## 5. False alarms and detection delay

The false-positive rate measures alerts outside the known episode. Detection delay counts how many faulty rows pass before the first correct alert; a value equal to the episode length means that episode was missed.

In [ ]:
operational = (
    detection.groupby(["split_label", "scenario_label"], as_index=False)[
        ["false_positive_rate", "detection_delay_rows"]
    ].mean()
)
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("False-positive rate", "Detection delay"),
    shared_yaxes=True,
    horizontal_spacing=0.08,
)
split_colors = {"Test 1": "#457b9d", "Test 2": "#e76f51"}
# Horizontal grouped bars are drawn bottom-to-top by trace order.
# Draw Test 2 first so Test 1 appears above it in every category.
for split in ["Test 2", "Test 1"]:
    rows = operational.loc[operational["split_label"] == split]
    fig.add_trace(
        go.Bar(
            x=rows["false_positive_rate"],
            y=rows["scenario_label"],
            orientation="h",
            name=split,
            legendgroup=split,
            marker_color=split_colors[split],
            hovertemplate="%{y}<br>False-positive rate: %{x:.2%}<extra></extra>",
        ),
        row=1, col=1,
    )
    delay_labels = rows.apply(
        lambda record: (
            "N/A — no fault"
            if record["scenario_label"] == SCENARIO_LABELS["clean"]
            else (
                "No delay" if record["detection_delay_rows"] == 0 else ""
            )
        ),
        axis=1,
    )
    delay_hover = rows.apply(
        lambda record: (
            "N/A — no fault"
            if record["scenario_label"] == SCENARIO_LABELS["clean"]
            else (
                "No delay"
                if record["detection_delay_rows"] == 0
                else f"{record['detection_delay_rows']:.1f} rows"
            )
        ),
        axis=1,
    )
    fig.add_trace(
        go.Bar(
            x=rows["detection_delay_rows"],
            y=rows["scenario_label"],
            orientation="h",
            name=split,
            legendgroup=split,
            marker_color=split_colors[split],
            text=delay_labels,
            textposition="outside",
            cliponaxis=False,
            customdata=[[value] for value in delay_hover],
            showlegend=False,
            hovertemplate="%{y}<br>Detection delay: %{customdata[0]}<extra></extra>",
        ),
        row=1, col=2,
    )
operational_order = list(SCENARIO_LABELS.values())
fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(operational_order)))
fig.update_yaxes(title_text="Simulated Light condition", row=1, col=1)
fig.update_yaxes(showticklabels=False, title_text=None, row=1, col=2)
fig.update_xaxes(title_text="Rate", tickformat=".0%", row=1, col=1)
fig.update_xaxes(
    title_text="Mean rows until detection", range=[0, 45], row=1, col=2
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Operational detector costs on held-out episodes",
    barmode="group",
    legend={"traceorder": "reversed"},
    height=650,
    margin={"l": 285},
)
fig.show()

**Conclusion.** Held-out false-positive rates of roughly 3–3.5% exceed the 1.15% training-validation result, indicating temporal distribution shift. Missing and extreme values have zero delay, while the 20-minute stuck rule necessarily reacts later and entirely misses some plausible constant or biased episodes.

## 6. End-to-end occupancy performance

The chart compares four strategies using mean full-period F1 across 15- and 60-minute episodes. Full-period metrics are intentionally used because they include both recovery during the fault and damage from false alarms elsewhere. Short episodes produce smaller changes than the earlier whole-period robustness interventions.

In [6]:
routing_summary = (
    routing.groupby(["split", "scenario_label", "strategy"], as_index=False)["f1"].mean()
)
strategy_order = ["fallback_only", "oracle_routing", "detector_routing", "primary_only"]
strategy_colors = {
    "primary_only": "#264653",
    "detector_routing": "#2a9d8f",
    "oracle_routing": "#e9c46a",
    "fallback_only": "#a8b3bd",
}
fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"), shared_yaxes=True)
for column, split in enumerate(["test_1", "test_2"], start=1):
    split_rows = routing_summary.loc[routing_summary["split"] == split]
    for strategy in strategy_order:
        rows = split_rows.loc[split_rows["strategy"] == strategy]
        fig.add_trace(
            go.Bar(
                x=rows["f1"],
                y=rows["scenario_label"],
                orientation="h",
                name=STRATEGY_LABELS[strategy],
                marker_color=strategy_colors[strategy],
                showlegend=column == 1,
                hovertemplate=(
                    STRATEGY_LABELS[strategy]
                    + "<br>%{y}<br>Mean full-period F1: %{x:.3f}<extra></extra>"
                ),
            ),
            row=1, col=column,
        )
fig.update_xaxes(range=[0.5, 1.01], title_text="Mean full-period F1")
routing_order = list(SCENARIO_LABELS.values())
fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(routing_order)))
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Primary, fallback, oracle, and causal-detector routing",
    barmode="group",
    legend={"traceorder": "reversed"},
    height=720,
    margin={"l": 285, "r": 35},
)
fig.show()

**Conclusion.** Detector routing recovers the Test 1 missing-episode loss, raising mean F1 from 0.956 to 0.971, equal to the oracle result. The benefit is not universal: in Test 2, false alarms send clean observations to the weaker fallback and slightly reduce F1 for several scenarios. Realistic routing therefore cannot be judged from fault recall alone.

## 7. Overall conclusion — no reliability recommendation yet

A causal detector can safely recognize explicit missingness and extreme values, but Light alone cannot reliably distinguish normal darkness from a sensor frozen at darkness. Plausible positive calibration bias is also missed. The increase in held-out false alarms shows that training-derived health thresholds drift across periods.

The detector preserves substantial oracle benefit in the clearest Test 1 failure, but unnecessary routing can reduce performance when the primary remains usable. The next Phase 5 experiment should compare fault-aware training and missingness indicators before writing the final reliability conclusion.